Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [121]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [122]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    # Genera N (size) punti casuali nel piano
    map = rng.random(size=(size, 2))
    # Inizializza la matrice dei pesi casuali
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            # se l'arco esiste distanza euclidea + rumore (dell'inizializzazione)
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            # se l'arco non esiste peso infinito
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


In [205]:
problem = create_problem(10, density=0.15, noise_level=10, negative_values=True)
problem

array([[    0.,    inf, 10280.,    inf,  6214.,    inf,    inf, -8731.,
        -6480.,    inf],
       [   inf,     0.,    inf,    inf,    inf, -5672.,    inf,    inf,
           inf,  3469.],
       [   inf,    inf,     0.,    inf,    inf,    inf,    inf,    inf,
           inf, -6395.],
       [   inf,    inf,  6432.,     0.,  4820.,    inf,    inf,    inf,
           inf,    inf],
       [ 4014.,    inf,    inf,  6016.,     0.,    inf,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,     0.,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,     0.,  -134.,
           inf,    inf],
       [   inf,    inf,  -251.,    inf,    inf,    inf,    inf,     0.,
           inf,    inf],
       [   inf,    inf,    inf, 10135.,  6151.,    inf,  -249.,    inf,
            0.,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,  5436.,  4947.,
           inf,     0.]])

In [201]:
## Bellman-Ford implementation, denied to negative cycles
from collections import deque


def distances_bellman_ford(problem: np.ndarray, start: int) -> tuple[list[int], list[float]]:
    size = problem.shape[0]
    distances = [np.inf] * size
    predecessors = [None] * size
    distances[start] = 0

    for _ in range(size - 1):
        updated = False
        for u in range(size):
            for v in range(size):
                #for each edge extract the weight
                weight = problem[u, v]
                if weight != np.inf and distances[u] + weight < distances[v] :
                    #if the distance from source to u plus w is less than the current distance from source to v, then update the distance of v
                    updated = True
                    predecessors[v] = u
                    distances[v] = distances[u] + weight
        if not updated:
            break

    # detect negative cycles 
    negative_cycles= False
    for u in range(size):
        for v in range(size):
            w = problem[u, v]
            if w != np.inf and distances[u] + w < distances[v]:
                negative_cycles=True
                break

    return predecessors, distances, negative_cycles
   


def shorthest_path_bellman_ford(predecessors, distances, end: int, negative_cycle) -> tuple[list[int], float]:
    #check if goal is reachable
    if distances[end] == np.inf:
        return None, np.inf
    # check if there is a negative cycle
    if negative_cycle:
        return None, -np.inf
    # there will always be a shorter path because of the negative cycle, cost= -inf

    path = []
    current_node = end 
    while current_node is not None:
        path.insert(0, current_node)
        current_node = predecessors[current_node]
    return path, distances[end]


predecessors, distances, negative_nodes = distances_bellman_ford(problem, 1)
path, cost = shorthest_path_bellman_ford(predecessors, distances, 2, negative_nodes)
print(path, cost)

None -inf


In [207]:
# Single problem test
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)
last_s=-1
predecessors, distances, negative_nodes = None, None, None
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        # cost = nx.path_weight(G, path, weight='weight')
        cost = nx.bellman_ford_path_length(G, s, d, weight='weight')
        path, cost
       
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    
    if s!=last_s:
        last_s = s
        predecessors, distances, negative_cycle = distances_bellman_ford(problem, s)
    p, c = shorthest_path_bellman_ford(predecessors, distances, d, negative_cycle)
    if path != p or cost != c:
        print("MISMATCH! path:", path, " computed:", p, " cost:", cost, " computed:", c )
    print("start:", s, " end:", d, " path:", path, " cost:", cost)
None

MISMATCH! path: None  computed: None  cost: -inf  computed: inf
start: 0  end: 1  path: None  cost: -inf
start: 0  end: 2  path: None  cost: -inf
start: 0  end: 3  path: None  cost: -inf
start: 0  end: 4  path: None  cost: -inf
MISMATCH! path: None  computed: None  cost: -inf  computed: inf
start: 0  end: 5  path: None  cost: -inf
start: 0  end: 6  path: None  cost: -inf
start: 0  end: 7  path: None  cost: -inf
start: 0  end: 8  path: None  cost: -inf
start: 0  end: 9  path: None  cost: -inf
start: 1  end: 2  path: None  cost: -inf
MISMATCH! path: None  computed: None  cost: -inf  computed: inf
start: 1  end: 3  path: None  cost: -inf
MISMATCH! path: None  computed: None  cost: -inf  computed: inf
start: 1  end: 4  path: None  cost: -inf
start: 1  end: 5  path: None  cost: -inf
start: 1  end: 6  path: None  cost: -inf
start: 1  end: 7  path: None  cost: -inf
MISMATCH! path: None  computed: None  cost: -inf  computed: inf
start: 1  end: 8  path: None  cost: -inf
start: 1  end: 9  path: 

In [208]:
size= [10, 20, 100, 200, 500]
density= 0.5
noise_level= 5.0
negative_values= [True, False]


for sz, neg in product(size, negative_values):
    problem = create_problem(sz, density=density, noise_level=noise_level, negative_values=neg)
    print("Problem size:", sz, " negative values:", neg)
    last_s=-1
    predecessors, distances, negative_nodes = None, None, None
    masked = np.ma.masked_array(problem, mask=np.isinf(problem))
    G = nx.from_numpy_array(masked, create_using=nx.DiGraph)
    for s, d in combinations(range(problem.shape[0]), 2):
        try:
            path = nx.bellman_ford_path(G, s, d, weight='weight')
            cost = nx.bellman_ford_path_length(G, s, d, weight='weight')
            path, cost
        except nx.NetworkXNoPath:
            # Nodes are not connected
            path = None
            cost = np.inf
        except nx.NetworkXUnbounded:
            # Negative cycle detected
            path = None
            cost = -np.inf
        
        if s!=last_s:
            last_s = s
            predecessors, distances, negative_cycle = distances_bellman_ford(problem, s)
        p, c = shorthest_path_bellman_ford(predecessors, distances, d, negative_cycle)
        if path != p or cost != c:
            print("MISMATCH! path:", path, " computed:", p, " cost:", cost, " computed:", c )
        print("start:", s, " end:", d, " path:", p, " cost:", c)


Problem size: 10  negative values: True
start: 0  end: 1  path: None  cost: -inf
start: 0  end: 2  path: None  cost: -inf
start: 0  end: 3  path: None  cost: -inf
start: 0  end: 4  path: None  cost: -inf
start: 0  end: 5  path: None  cost: -inf
start: 0  end: 6  path: None  cost: -inf
start: 0  end: 7  path: None  cost: -inf
start: 0  end: 8  path: None  cost: -inf
start: 0  end: 9  path: None  cost: -inf
start: 1  end: 2  path: None  cost: -inf
start: 1  end: 3  path: None  cost: -inf
start: 1  end: 4  path: None  cost: -inf
start: 1  end: 5  path: None  cost: -inf
start: 1  end: 6  path: None  cost: -inf
start: 1  end: 7  path: None  cost: -inf
start: 1  end: 8  path: None  cost: -inf
start: 1  end: 9  path: None  cost: -inf
start: 2  end: 3  path: None  cost: -inf
start: 2  end: 4  path: None  cost: -inf
start: 2  end: 5  path: None  cost: -inf
start: 2  end: 6  path: None  cost: -inf
start: 2  end: 7  path: None  cost: -inf
start: 2  end: 8  path: None  cost: -inf
start: 2  end: 9 

KeyboardInterrupt: 